# 00 – Transform Firewall Export Logs (CSV) → Parquet

This notebook demonstrates how to:
1. Parse the `;`-separated firewall export log using `FirewallExportParser`
2. Inspect the structured DataFrame
3. Filtrer sur la plage de dates et supprimer la colonne `fw`
4. Persist to **Parquet** format
5. Reload and verify

Format attendu (pas d'en-tête, séparateur `;`) :
```
2025-03-20 01:29:24;94.102.61.47;159.84.146.99;TCP;52502;3178;999;DENY;eth0;;6
```
Colonnes : `timestamp ; src_ip ; dst_ip ; proto ; src_port ; dst_port ; rule ; action ; interface_in ; interface_out ; fw`

In [17]:
import sys
from pathlib import Path

# Make the project root importable when running from the notebooks/ folder
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\cyraptor\Documents\PROJECTS\Challenge-CyberSecu-SISE


## 1 – Configuration des chemins

Renseignez ici le chemin vers votre fichier `.log` exporté (format `;` séparé) et le chemin de sortie Parquet.

Le filtre de dates et la suppression de `fw` sont configurables via les variables ci-dessous.

In [18]:
from pathlib import Path

# ── À MODIFIER ────────────────────────────────────────────────────────────────
LOG_PATH     = "C:/Users/cyraptor/Documents/PROJECTS/Challenge-CyberSecu-SISE/data/raw/test.csv"   # chemin du fichier export CSV
PARQUET_PATH = PROJECT_ROOT / "data" / "processed" / "firewall_export.parquet"  # sortie .parquet

# Filtre de dates (optionnel) — mettre FILTER_DATES = False pour désactiver
FILTER_DATES = True
DATE_FROM    = "2025-11-01"   # inclus
DATE_TO      = "2026-02-28"   # inclus
# ──────────────────────────────────────────────────────────────────────────────

# Vérifie que le fichier source existe
if not Path(LOG_PATH).exists():
    raise FileNotFoundError(f"Fichier introuvable : {LOG_PATH}")

print(f"Source  : {LOG_PATH}")
print(f"Sortie  : {PARQUET_PATH}")
if FILTER_DATES:
    print(f"Filtre  : {DATE_FROM}  →  {DATE_TO}")

Source  : C:/Users/cyraptor/Documents/PROJECTS/Challenge-CyberSecu-SISE/data/raw/test.csv
Sortie  : C:\Users\cyraptor\Documents\PROJECTS\Challenge-CyberSecu-SISE\data\processed\firewall_export.parquet
Filtre  : 2025-11-01  →  2026-02-28


## 2 – Parser le fichier export avec `FirewallExportParser`

Utilisable aussi directement dans l'application :

```python
from parsers import FirewallExportParser
# ou via la factory :
from parsers import ParserFactory
parser = ParserFactory.create("firewall_export")
df = parser.parse("/chemin/vers/log_export.log")
```

In [19]:
from parsers import FirewallExportParser

parser = FirewallExportParser()

df = parser.parse(str(LOG_PATH))

is_valid, errors = parser.validate(df)
print(f"Valid: {is_valid}  |  Errors: {errors or 'none'}")
print(f"\nShape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df

Valid: True  |  Errors: none

Shape: (10, 11)
Columns: ['timestamp', 'src_ip', 'dst_ip', 'proto', 'src_port', 'dst_port', 'rule', 'action', 'interface_in', 'interface_out', 'fw']


,timestamp,src_ip,dst_ip,proto,src_port,dst_port,rule,action,interface_in,interface_out,fw
0,2025-11-02 01:29:24,94.102.61.47,159.84.146.99,TCP,52502,3178,999,DENY,eth0,NaN,6
1,2025-11-03 01:29:25,176.111.174.85,159.84.146.99,TCP,48739,2231,999,DENY,eth0,NaN,6
2,2025-11-04 01:29:27,66.249.65.106,159.84.146.99,TCP,50501,443,1,PERMIT,eth0,NaN,6
3,2025-11-05 01:29:34,89.248.163.75,159.84.146.99,TCP,43312,8845,999,DENY,eth0,NaN,6
4,2025-11-06 01:29:38,42.58.163.244,159.84.146.99,TCP,9746,23,7,DENY,eth0,NaN,6
5,2025-11-07 01:29:39,173.252.83.5,159.84.146.99,TCP,50908,443,1,PERMIT,eth0,NaN,6
6,2025-11-08 01:29:41,66.249.65.106,159.84.146.99,TCP,45634,443,1,PERMIT,eth0,NaN,6
7,2025-11-09 01:29:48,66.249.65.106,159.84.146.99,TCP,40785,443,1,PERMIT,eth0,NaN,6
8,2025-11-10 01:29:53,34.30.180.51,159.84.146.99,TCP,59756,443,1,PERMIT,eth0,NaN,6
9,2025-11-11 01:29:58,85.217.144.149,159.84.146.99,TCP,43267,33944,999,DENY,eth0,NaN,6


In [20]:
# Inspect dtypes and a quick summary
print(df.dtypes)
print()
df[["timestamp", "action", "src_ip", "dst_ip", "src_port", "dst_port", "rule"]].head(10)

timestamp        datetime64[ns]
src_ip                   object
dst_ip                   object
proto                  category
src_port                  Int64
dst_port                  Int64
rule                      Int64
action                 category
interface_in             object
interface_out           float64
fw                        Int64
dtype: object



,timestamp,action,src_ip,dst_ip,src_port,dst_port,rule
0,2025-11-02 01:29:24,DENY,94.102.61.47,159.84.146.99,52502,3178,999
1,2025-11-03 01:29:25,DENY,176.111.174.85,159.84.146.99,48739,2231,999
2,2025-11-04 01:29:27,PERMIT,66.249.65.106,159.84.146.99,50501,443,1
3,2025-11-05 01:29:34,DENY,89.248.163.75,159.84.146.99,43312,8845,999
4,2025-11-06 01:29:38,DENY,42.58.163.244,159.84.146.99,9746,23,7
5,2025-11-07 01:29:39,PERMIT,173.252.83.5,159.84.146.99,50908,443,1
6,2025-11-08 01:29:41,PERMIT,66.249.65.106,159.84.146.99,45634,443,1
7,2025-11-09 01:29:48,PERMIT,66.249.65.106,159.84.146.99,40785,443,1
8,2025-11-10 01:29:53,PERMIT,34.30.180.51,159.84.146.99,59756,443,1
9,2025-11-11 01:29:58,DENY,85.217.144.149,159.84.146.99,43267,33944,999


## 3 – Post-traitement : filtre de dates & suppression de `fw`

Conformément aux consignes :
- la colonne `fw` (numéro du firewall) est **supprimée**
- seules les lignes entre **novembre 2025** et **février 2026** sont conservées (si `FILTER_DATES = True`)

In [21]:
df_out = df.copy()

# ── Distribution des valeurs FW (informatif) ───────────────────────────────
print("Distribution des valeurs FW (toutes conservées) :")
print(df_out["fw"].value_counts().sort_index().to_string())
print()

# ── Suppression de la colonne fw ───────────────────────────────────────────
df_out = df_out.drop(columns=["fw", "interface_out"], errors="ignore")

# ── Filtre de dates (optionnel) ────────────────────────────────────────────
if FILTER_DATES:
    before = len(df_out)
    mask   = (df_out["timestamp"] >= DATE_FROM) & (df_out["timestamp"] <= DATE_TO)
    df_out = df_out.loc[mask].reset_index(drop=True)
    print(f"Filtre dates : {before:,} → {len(df_out):,} lignes  ({DATE_FROM} → {DATE_TO})")
else:
    print(f"Filtre de dates désactivé — {len(df_out):,} lignes conservées")

print(f"\nColonnes : {list(df_out.columns)}")
df_out.head(5)

Distribution des valeurs FW (toutes conservées) :
fw
6    10

Filtre dates : 10 → 10 lignes  (2025-11-01 → 2026-02-28)

Colonnes : ['timestamp', 'src_ip', 'dst_ip', 'proto', 'src_port', 'dst_port', 'rule', 'action', 'interface_in']


,timestamp,src_ip,dst_ip,proto,src_port,dst_port,rule,action,interface_in
0,2025-11-02 01:29:24,94.102.61.47,159.84.146.99,TCP,52502,3178,999,DENY,eth0
1,2025-11-03 01:29:25,176.111.174.85,159.84.146.99,TCP,48739,2231,999,DENY,eth0
2,2025-11-04 01:29:27,66.249.65.106,159.84.146.99,TCP,50501,443,1,PERMIT,eth0
3,2025-11-05 01:29:34,89.248.163.75,159.84.146.99,TCP,43312,8845,999,DENY,eth0
4,2025-11-06 01:29:38,42.58.163.244,159.84.146.99,TCP,9746,23,7,DENY,eth0


## 4 – Sauvegarde en Parquet

`save_parquet` et `load_parquet` sont dans `utils.io` et peuvent être importés partout dans l'application.

In [22]:
from utils.io import save_parquet, load_parquet

saved = save_parquet(df_out, PARQUET_PATH, compression="snappy")
print(f"Sauvegardé → {saved}  ({saved.stat().st_size:,} bytes)")

Sauvegardé → C:\Users\cyraptor\Documents\PROJECTS\Challenge-CyberSecu-SISE\data\processed\firewall_export.parquet  (6,145 bytes)


## 5 – Reload & vérification

In [23]:
df_loaded = load_parquet(PARQUET_PATH)

print(f"Rows loaded: {len(df_loaded)}")
print(f"Dtypes preserved: {dict(df_loaded.dtypes)}")
df_loaded

Rows loaded: 10
Dtypes preserved: {'timestamp': dtype('<M8[ns]'), 'src_ip': dtype('O'), 'dst_ip': dtype('O'), 'proto': CategoricalDtype(categories=['TCP'], ordered=False, categories_dtype=object), 'src_port': Int64Dtype(), 'dst_port': Int64Dtype(), 'rule': Int64Dtype(), 'action': CategoricalDtype(categories=['DENY', 'PERMIT'], ordered=False, categories_dtype=object), 'interface_in': dtype('O')}


,timestamp,src_ip,dst_ip,proto,src_port,dst_port,rule,action,interface_in
0,2025-11-02 01:29:24,94.102.61.47,159.84.146.99,TCP,52502,3178,999,DENY,eth0
1,2025-11-03 01:29:25,176.111.174.85,159.84.146.99,TCP,48739,2231,999,DENY,eth0
2,2025-11-04 01:29:27,66.249.65.106,159.84.146.99,TCP,50501,443,1,PERMIT,eth0
3,2025-11-05 01:29:34,89.248.163.75,159.84.146.99,TCP,43312,8845,999,DENY,eth0
4,2025-11-06 01:29:38,42.58.163.244,159.84.146.99,TCP,9746,23,7,DENY,eth0
5,2025-11-07 01:29:39,173.252.83.5,159.84.146.99,TCP,50908,443,1,PERMIT,eth0
6,2025-11-08 01:29:41,66.249.65.106,159.84.146.99,TCP,45634,443,1,PERMIT,eth0
7,2025-11-09 01:29:48,66.249.65.106,159.84.146.99,TCP,40785,443,1,PERMIT,eth0
8,2025-11-10 01:29:53,34.30.180.51,159.84.146.99,TCP,59756,443,1,PERMIT,eth0
9,2025-11-11 01:29:58,85.217.144.149,159.84.146.99,TCP,43267,33944,999,DENY,eth0
